# Feature pipeline — Premier League match prediction

This notebook builds `games_with_features.csv` end-to-end from public sources. Run it once before opening the final analysis notebook.

**Inputs (fetched live):**
- 10 seasons of Premier League match results and Bet365 odds from `football-data.co.uk`
- Club Elo ratings from `clubelo.com` (one API call per club)

**Output:** a single CSV with 19 model features per match, plus the raw match record (date, teams, goals, full-time result, Bet365 odds, referee, cards). All rolling features apply `.shift(1)` so the current match is never used to predict itself.

The whole pipeline runs in roughly two minutes, dominated by the clubelo.com calls.

## 1. Setup

In [2]:
import warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import requests
from io import StringIO
from pathlib import Path

# Configure the output path. On Colab with Drive mounted, point this at your
# project folder (e.g. /content/drive/MyDrive/PL_project/data/games_with_features.csv).
OUTPUT_PATH = Path("data/games_with_features.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Will save to: {OUTPUT_PATH.resolve()}")

Will save to: /content/data/games_with_features.csv


## 2. Load 10 seasons of match results

`football-data.co.uk` publishes one CSV per Premier League season at a stable URL. We pull seasons 2015/16 through 2024/25 and concatenate them.

In [3]:
SEASONS = list(range(2015, 2025))  # season-start years: 2015/16, ..., 2024/25
BASE_URL = "https://www.football-data.co.uk/mmz4281"

def load_season(start_year):
    short = f"{str(start_year)[-2:]}{str(start_year+1)[-2:]}"  # 1516, 1617, ...
    url = f"{BASE_URL}/{short}/E0.csv"
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    df = pd.read_csv(StringIO(r.text))
    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
    df["Season"] = start_year
    return df

all_seasons = pd.concat([load_season(y) for y in SEASONS], ignore_index=True)
all_seasons = all_seasons.dropna(subset=["FTR"]).sort_values("Date").reset_index(drop=True)
print(f"Loaded {len(all_seasons):,} matches across {len(SEASONS)} seasons "
      f"({all_seasons.Date.min().date()} – {all_seasons.Date.max().date()})")
print(f"\nMatches per season:")
print(all_seasons.groupby("Season").size())

Loaded 3,800 matches across 10 seasons (2015-08-08 – 2025-05-25)

Matches per season:
Season
2015    380
2016    380
2017    380
2018    380
2019    380
2020    380
2021    380
2022    380
2023    380
2024    380
dtype: int64


## 3. Rolling form features (last 5 matches)

For each team, compute its goals-for, goals-against, and points averaged over its previous five matches. `.shift(1)` ensures the current match never appears in its own rolling window.

In [4]:
def build_team_form(df_in, n=5):
    """Long-form per-team match record with rolling means."""
    home = df_in[["Date","HomeTeam","FTHG","FTAG","FTR"]].copy()
    home.columns = ["Date","Team","GF","GA","Result"]
    home["Pts"]    = home["Result"].map({"H":3,"D":1,"A":0})
    home["IsHome"] = 1

    away = df_in[["Date","AwayTeam","FTAG","FTHG","FTR"]].copy()
    away.columns = ["Date","Team","GF","GA","Result"]
    away["Pts"]    = away["Result"].map({"H":0,"D":1,"A":3})
    away["IsHome"] = 0

    long = pd.concat([home, away]).sort_values(["Team","Date"]).reset_index(drop=True)
    for col in ["GF","GA","Pts"]:
        long[f"{col}_last{n}"] = (
            long.groupby("Team")[col].rolling(n, min_periods=1).mean()
                .shift(1).reset_index(level=0, drop=True)
        )
    return long

form = build_team_form(all_seasons)

# Pivot back: home-team features and away-team features for each match
hf = form[form["IsHome"] == 1][["Date","Team","GF_last5","GA_last5","Pts_last5"]]
hf.columns = ["Date","HomeTeam","Home_GF_last5","Home_GA_last5","Home_Pts_last5"]
af = form[form["IsHome"] == 0][["Date","Team","GF_last5","GA_last5","Pts_last5"]]
af.columns = ["Date","AwayTeam","Away_GF_last5","Away_GA_last5","Away_Pts_last5"]

df = all_seasons.merge(hf, on=["Date","HomeTeam"], how="left")
df = df.merge(af, on=["Date","AwayTeam"], how="left")

# Home–Away differentials — often the strongest single predictors
df["GF_diff_last5"]  = df["Home_GF_last5"]  - df["Away_GF_last5"]
df["GA_diff_last5"]  = df["Home_GA_last5"]  - df["Away_GA_last5"]
df["Pts_diff_last5"] = df["Home_Pts_last5"] - df["Away_Pts_last5"]
print(f"Rolling form features added.   Non-null rows: {df['Pts_diff_last5'].notna().sum():,}")

Rolling form features added.   Non-null rows: 3,799


## 4. Rest days

For each match, how many days since each team's previous match — captures fatigue and midweek-fixture effects.

In [5]:
# Build a dict: team -> sorted array of all dates that team played
team_to_dates = {}
combined = pd.concat([
    all_seasons[["Date","HomeTeam"]].rename(columns={"HomeTeam":"Team"}),
    all_seasons[["Date","AwayTeam"]].rename(columns={"AwayTeam":"Team"}),
]).sort_values("Date")
for team, grp in combined.groupby("Team"):
    team_to_dates[team] = grp["Date"].values

def rest_for(team, match_date):
    arr = team_to_dates.get(team)
    if arr is None:
        return np.nan
    mdate = np.datetime64(match_date)
    mask = arr < mdate
    if not mask.any():
        return np.nan
    return (match_date - pd.Timestamp(arr[mask].max())).days

df["Home_rest_days"] = [rest_for(t, d) for t, d in zip(df["HomeTeam"], df["Date"])]
df["Away_rest_days"] = [rest_for(t, d) for t, d in zip(df["AwayTeam"], df["Date"])]
df["Rest_diff"]      = df["Home_rest_days"] - df["Away_rest_days"]
print(f"Rest-day features added.")

Rest-day features added.


## 5. Cumulative season points

Each team's running league points so far **this season**, going into the match. Captures season-long standing beyond the last five matches. Resets at every season boundary.

In [6]:
def add_season_points(matches_df):
    matches_df = matches_df.sort_values("Date").copy()
    season_tally = {}  # (season, team) -> running points
    home_pts, away_pts = [], []
    for _, row in matches_df.iterrows():
        kh = (row["Season"], row["HomeTeam"])
        ka = (row["Season"], row["AwayTeam"])
        home_pts.append(season_tally.get(kh, 0))
        away_pts.append(season_tally.get(ka, 0))
        if   row["FTR"] == "H": season_tally[kh] = season_tally.get(kh, 0) + 3
        elif row["FTR"] == "A": season_tally[ka] = season_tally.get(ka, 0) + 3
        elif row["FTR"] == "D":
            season_tally[kh] = season_tally.get(kh, 0) + 1
            season_tally[ka] = season_tally.get(ka, 0) + 1
    matches_df["Home_SeasonPts"] = home_pts
    matches_df["Away_SeasonPts"] = away_pts
    matches_df["SeasonPts_diff"] = matches_df["Home_SeasonPts"] - matches_df["Away_SeasonPts"]
    return matches_df

df = add_season_points(df)
print(f"Season-points features added.")

Season-points features added.


## 6. Club Elo

We pull each club's Elo history from `clubelo.com` (one HTTP call per club, cached in memory) and look up the team's Elo as of each match date with a simple as-of join.

Football-data.co.uk and clubelo.com use slightly different club name conventions, so we keep a small translation map for the cases where they differ.

In [7]:
ELO_NAME_MAP = {
    "Man United":    "ManUnited",
    "Man City":      "ManCity",
    "Nott'm Forest": "Forest",
    "Sheffield Utd": "SheffieldUnited",
    "West Brom":     "WestBrom",
    "West Ham":      "WestHam",
    "Crystal Palace":"CrystalPalace",
    "Aston Villa":   "AstonVilla",
    "Leicester":     "Leicester",
    "Norwich":       "Norwich",
    "Swansea":       "Swansea",
    # Most teams (Arsenal, Chelsea, Liverpool, Tottenham, etc.) match directly.
}

def clubelo_name(team):
    return ELO_NAME_MAP.get(team, team.replace(" ", ""))

teams = sorted(set(df["HomeTeam"]).union(df["AwayTeam"]))
print(f"Fetching Elo history for {len(teams)} teams...")

elo_history = {}
for team in teams:
    api_name = clubelo_name(team)
    try:
        r = requests.get(f"http://api.clubelo.com/{api_name}", timeout=15)
        if r.status_code == 200 and len(r.text) > 100:
            team_df = pd.read_csv(StringIO(r.text))
            if len(team_df) > 0 and "Elo" in team_df.columns:
                team_df["From"] = pd.to_datetime(team_df["From"])
                elo_history[team] = team_df.sort_values("From").reset_index(drop=True)
    except Exception as e:
        print(f"  failed for {team}: {e}")
    time.sleep(0.4)  # be polite to the free API

print(f"Got Elo history for {len(elo_history)}/{len(teams)} teams.")

Fetching Elo history for 34 teams...
Got Elo history for 34/34 teams.


In [8]:
def lookup_elo(team, match_date):
    hist = elo_history.get(team)
    if hist is None or len(hist) == 0:
        return np.nan
    prior = hist[hist["From"] <= match_date]
    if len(prior) == 0:
        return np.nan
    return float(prior.iloc[-1]["Elo"])

df["Home_Elo"] = [lookup_elo(t, d) for t, d in zip(df["HomeTeam"], df["Date"])]
df["Away_Elo"] = [lookup_elo(t, d) for t, d in zip(df["AwayTeam"], df["Date"])]
df["Elo_diff"] = df["Home_Elo"] - df["Away_Elo"]

missing = df["Elo_diff"].isna().sum()
print(f"Elo features added.   Matches with missing Elo: {missing}")

Elo features added.   Matches with missing Elo: 0


## 7. COVID no-fans flag

The 2020/21 season was played entirely without crowds. Home advantage collapsed during this window; a binary indicator lets the model adjust.

In [9]:
no_fans_start = pd.Timestamp("2020-06-17")
no_fans_end   = pd.Timestamp("2021-05-15")
df["no_fans"] = ((df["Date"] >= no_fans_start) & (df["Date"] <= no_fans_end)).astype(int)
print(f"COVID no-fans flag set on {df['no_fans'].sum()} matches.")

COVID no-fans flag set on 448 matches.


## 8. Save

The output file is what the final analysis notebook loads. It contains the original football-data.co.uk columns (results, Bet365 odds, referee, cards, etc.) plus the 19 engineered features.

In [10]:
# Sanity check: list what's in the final DataFrame
feature_cols = [
    "Home_GF_last5","Home_GA_last5","Home_Pts_last5",
    "Away_GF_last5","Away_GA_last5","Away_Pts_last5",
    "GF_diff_last5","GA_diff_last5","Pts_diff_last5",
    "Home_rest_days","Away_rest_days","Rest_diff",
    "Home_SeasonPts","Away_SeasonPts","SeasonPts_diff",
    "Home_Elo","Away_Elo","Elo_diff",
    "no_fans",
]
present = [c for c in feature_cols if c in df.columns]
print(f"Feature columns present: {len(present)}/19")
print(f"Total rows: {len(df):,}")
print(f"Rows with no missing feature: {df[present].notna().all(axis=1).sum():,}")

df.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved {len(df):,} rows × {len(df.columns)} cols to {OUTPUT_PATH}")

Feature columns present: 19/19
Total rows: 3,800
Rows with no missing feature: 3,776

Saved 3,800 rows × 174 cols to data/games_with_features.csv


## Next step

Open `final_notebook.ipynb` and point its `DATA_PATH` at the file written above. The analysis runs end-to-end from there.